In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
%matplotlib inline
%matplotlib qt
import matplotlib.pyplot as plt
import torch.optim as optim


In [2]:
# Load FashionMNIST dataset
# -----------------------------------
# -----------------------------------
# Load FashionMNIST dataset
# -----------------------------------
train_set = torchvision.datasets.FashionMNIST(
    root="./data/FashionMNIST",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# -----------------------------------
# Access an item by index
# -----------------------------------
index = 0

image, label = train_set[index]

# -----------------------------------
# Print info
# -----------------------------------
print("Index:", index)
print("Label:", label)
print("Tensor shape:", image.shape)

Index: 0
Label: 9
Tensor shape: torch.Size([1, 28, 28])


In [3]:
#torch.set_printoptions(linewidth=120)
torch.set_grad_enabled(True)

torch.autograd.grad_mode.set_grad_enabled(mode=True)

In [4]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

class NetworkA(nn.Module):
    def __init__(self):
        super(Network, self).__init()
        self.layer = None
    def forward(self, t):
        t = self.layer(t)
        return t

class Network(nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        
        #Layer takes one input, has a kernel/filter of size 5 and outputs 5 featuremaps
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        #Next layer taking in input of val 1
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        
        self.fc1 = nn.Linear(in_features=12*4*4, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=60)
        self.out = nn.Linear(in_features=60, out_features=10)

    def forward(self, t):
        #Implement forward pass
        t=t
        
        t=self.conv1(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        t=self.conv2(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        
        t=t.reshape(-1,12 * 4 * 4)
        t=self.fc1(t)
        t=F.relu(t)
        
        t=self.fc2(t)
        t=F.relu(t)
        
        t=self.out(t)
       # t=F.softmax(t, dim=1)
        
        return t

In [5]:
network = Network()

In [6]:
train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=100,
    shuffle=True
)
optimizer = optim.Adam(network.parameters(), lr=0.01)

In [7]:
batch = next(iter(train_loader))
images, labels = batch
#Image shape: torch.Size([10, 1, 28, 28]) --> 10 28x28 images with one color channel
#Label Shape: torch.Size([10]) -->one label for each image

In [8]:
#Pred Shape: torch.Size([10, 10]) --> Two axis each length 10, ten images with 10 prediction classes

#Argmax to check which index has highest prediction value == compare with label after
#pred.argmax(dim=1)

#Compares with the label and gives 1 or 0 for match & calling sum reduces the output into a single number of correct predictions
#pred.argmax(dim=1).eq(labels).sum()

#Same thing but uses item to get the number of correct predictions
def get_num_correct(prediction, label):
    return prediction.argmax(dim=1).eq(label).sum().item()

In [9]:
pred = network(images)
loss = F.cross_entropy(pred, labels)
print(loss.requires_grad)
print(loss.grad_fn)
loss.backward()
optimizer.step() #updates the weights based off gradient == step by step

print('Loss 1', loss.item())
preds = network(images)
loss = F.cross_entropy(preds, labels)
print('Loss 2', loss.item())

True
Loss 1 2.290656089782715
Loss 2 2.265061855316162


In [10]:
#Confusion Matrix
#Access labels for targets == get predcitions for entire set

def get_all_predictions(model, loader):
    all_predictions = torch.tensor([])
    for batch in loader:
        images, labels = batch
        
        predictions = model(images)
        all_predictions = torch.cat(
            (all_predictions, predictions)
            , dim=0
        )
    return all_predictions
#Predictions W/out gradients
with torch.no_grad():
    prediction_loader = torch.utils.data.DataLoader(train_set, batch_size=10000)
    train_preds = get_all_predictions(network, prediction_loader)

print(train_preds.requires_grad)

preds_correct = get_num_correct(train_preds, train_set.targets)
print('Correct:', preds_correct)
print('Accuracy:', preds_correct/len(train_set))

False
Correct: 6000
Accuracy: 0.1


In [11]:
train_preds.argmax(dim=1)
stacked = torch.stack(
    (
        train_set.targets
        , train_preds.argmax(dim=1)
    )
    ,dim=1
)

cmt = torch.zeros(10,10, dtype=torch.int32)
cmt

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)

In [12]:
for p in stacked:
    j,k = p.tolist()
    cmt[j, k] = cmt[j, k] + 1
cmt

tensor([[   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0, 6000,    0,    0,    0,    0]],
       dtype=torch.int32)

In [13]:
from torch.utils.tensorboard import SummaryWriter

tb= SummaryWriter()
network=Network()
images, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(images)

tb.add_image('images', grid)
tb.add_graph(network, images)
tb.close()

In [21]:
#Training loop
batch_size = 100
lr = 0.01
network = Network()
training_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size)
optimizer = optim.Adam(network.parameters(), lr=lr)

images, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(images)

comment = f'batch_size={batch_size}, lr = {lr}'
tb= SummaryWriter(comment=comment)
tb.add_image('images', grid)
tb.add_graph(network, images)

for epoch in range(1):
    totalLoss = 0
    totalCorrect = 0
    
    for abtch in training_loader:
        images, labels = batch
        
        pred = network(images)
        loss = F.cross_entropy(pred, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        totalLoss += loss.item() * batch_size
        totalCorrect += get_num_correct(pred, labels)
    
    tb.add_scalar('Loss:', totalLoss, epoch)
    tb.add_scalar('Correct:', totalCorrect, epoch)
    tb.add_scalar('Accuracy:', totalCorrect/len(train_set), epoch)
    
    tb.add_histogram('conv1.bias', network.conv1.bias, epoch)
    tb.add_histogram('conv1.weight',network.conv1.weight, epoch)
    tb.add_histogram('conv1.weight.grad', network.conv1.weight.grad, epoch)
    print("epoch:",epoch,"Total Correct: ", totalCorrect, "Loss: ", totalLoss)

epoch: 0 Total Correct:  58378 Loss:  4264.021994064569


In [22]:
from itertools import product

parameters = dict(
    lr=[.01,.001]
    ,batch_size= [10,100,1000]
    ,shuffle = [True,False]
)

paramValues = [v for v in parameters.values()]
paramValues

[[0.01, 0.001], [10, 100, 1000], [True, False]]

In [23]:
for lr, batch_size, shuffle in product(*paramValues):
    print (lr, batch_size, shuffle)

0.01 10 True
0.01 10 False
0.01 100 True
0.01 100 False
0.01 1000 True
0.01 1000 False
0.001 10 True
0.001 10 False
0.001 100 True
0.001 100 False
0.001 1000 True
0.001 1000 False
